# MVP — Pipeline de Dados na Nuvem
## IDH, PIB per capita e Pegada Ecológica

Esta versão prioriza **análises realizadas diretamente sobre tabelas persistidas no Databricks**.

**Princípio do projeto:** Python/PySpark é utilizado para ingestão, transformação, validação e construção das tabelas. As análises finais são executadas majoritariamente com **Spark SQL sobre tabelas Delta persistidas**, incluindo tabelas auxiliares na camada Gold.

Fluxo:

`CSV/GitHub → Bronze → Silver → Gold detalhada → Gold auxiliares → SQL analítico`

## 1. Objetivo e perguntas

**Objetivo geral:** construir um pipeline em nuvem para identificar relações entre HDI e PIB per capita e quatro componentes da pegada ecológica: carbono, pesca, cultivo e pastagem.

Perguntas:
1. Qual é a relação entre HDI e Carbon Footprint?
2. Qual é a relação entre GDP per Capita e Carbon Footprint?
3. Como HDI e GDP per Capita se relacionam com Fish Footprint?
4. Como se relacionam com Cropland Footprint?
5. Como se relacionam com Grazing Footprint?
6. Como os indicadores se comportam por região?
7. Quais países aparecem como valores extremos?
8. Quais relações são mais fortes segundo as correlações calculadas sobre a tabela Gold?

A análise é associativa; correlação não demonstra causalidade.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

DATA_SOURCE_URL = "https://raw.githubusercontent.com/tfiorio/pos-pucrio/refs/heads/sprint03/countries.csv"
GITHUB_REPOSITORY_URL = "https://github.com/tfiorio/pos-pucrio/tree/sprint03"

CATALOG = "workspace"
SCHEMA = "mvp_ecological_footprint"

BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.countries_bronze"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.countries_silver"

GOLD_TABLE = f"{CATALOG}.{SCHEMA}.countries_environmental_analysis"
GOLD_REGION_TABLE = f"{CATALOG}.{SCHEMA}.analysis_by_region"
GOLD_HDI_TABLE = f"{CATALOG}.{SCHEMA}.analysis_by_hdi_band"
GOLD_GDP_TABLE = f"{CATALOG}.{SCHEMA}.analysis_by_gdp_band"
GOLD_CORR_TABLE = f"{CATALOG}.{SCHEMA}.analysis_correlations"
GOLD_OUTLIER_TABLE = f"{CATALOG}.{SCHEMA}.analysis_outliers"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("Schema pronto:", f"{CATALOG}.{SCHEMA}")

Schema pronto: workspace.mvp_ecological_footprint


## 2. Ingestão e Bronze

A Bronze preserva os dados recebidos. São acrescentados apenas metadados técnicos.

In [0]:
import pandas as pd

df_pd = pd.read_csv(DATA_SOURCE_URL)
df_raw = spark.createDataFrame(df_pd)

df_bronze = (
    df_raw
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_system", F.lit("GitHub RAW / CSV"))
    .withColumn("source_file", F.lit("countries.csv"))
)

(df_bronze.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true")
 .option("delta.columnMapping.mode","name")
 .option("delta.minReaderVersion","2")
 .option("delta.minWriterVersion","5")
 .saveAsTable(BRONZE_TABLE))

print("Bronze:", BRONZE_TABLE, "| registros:", spark.table(BRONZE_TABLE).count())
display(spark.table(BRONZE_TABLE).limit(10))

Bronze: workspace.mvp_ecological_footprint.countries_bronze | registros: 188


Country,Region,Population (millions),HDI,GDP per Capita,Cropland Footprint,Grazing Footprint,Forest Footprint,Carbon Footprint,Fish Footprint,Total Ecological Footprint,Cropland,Grazing Land,Forest Land,Fishing Water,Urban Land,Total Biocapacity,Biocapacity Deficit or Reserve,Earths Required,Countries Required,Data Quality,ingestion_timestamp,source_system,source_file
Afghanistan,Middle East/Central Asia,29.82,0.46,$614.66,0.3,0.2,0.08,0.18,0.0,0.79,0.24,0.2,0.02,0.0,0.04,0.5,-0.3,0.46,1.6,6,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv
Albania,Northern/Eastern Europe,3.16,0.73,"$4,534.37",0.78,0.22,0.25,0.87,0.02,2.21,0.55,0.21,0.29,0.07,0.06,1.18,-1.03,1.27,1.87,6,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv
Algeria,Africa,38.48,0.73,"$5,430.57",0.6,0.16,0.17,1.14,0.01,2.12,0.24,0.27,0.03,0.01,0.03,0.59,-1.53,1.22,3.61,5,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv
Angola,Africa,20.82,0.52,"$4,665.91",0.33,0.15,0.12,0.2,0.09,0.93,0.2,1.42,0.64,0.26,0.04,2.55,1.61,0.54,0.37,6,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv
Antigua and Barbuda,Latin America,0.09,0.78,"$13,205.10",null,null,null,null,null,5.38,null,null,null,null,null,0.94,-4.44,3.11,5.7,2,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv
Argentina,Latin America,41.09,0.83,"$13,540.00",0.78,0.79,0.29,1.08,0.1,3.14,2.64,1.86,0.66,1.67,0.1,6.92,3.78,1.82,0.45,6,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv
Armenia,Middle East/Central Asia,2.97,0.73,"$3,426.39",0.74,0.18,0.34,0.89,0.01,2.23,0.44,0.26,0.1,0.02,0.07,0.89,-1.35,1.29,2.52,3B,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv
Aruba,Latin America,0.1,null,null,null,null,null,null,null,11.88,null,null,null,null,null,0.57,-11.31,6.86,20.69,2,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv
Australia,Asia-Pacific,23.05,0.93,"$66,604.20",2.68,0.63,0.89,4.85,0.11,9.31,5.42,5.81,2.01,3.19,0.14,16.57,7.26,5.37,0.56,5,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv
Austria,European Union,8.46,0.88,"$51,274.10",0.82,0.27,0.63,4.14,0.06,6.06,0.71,0.16,2.04,0.0,0.15,3.07,-3.0,3.5,1.98,5,2026-09-23T13:19:06.794Z,GitHub RAW / CSV,countries.csv


## 3. Qualidade inicial — consultando a Bronze persistida

A partir daqui, mesmo as verificações de qualidade partem da tabela salva no Databricks, e não do DataFrame de ingestão.

In [0]:
bronze = spark.table(BRONZE_TABLE)

null_exprs = [
    F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1).otherwise(0)).alias(c)
    for c in df_raw.columns
]
display(bronze.select(null_exprs))

print("Registros Bronze:", bronze.count())
print("Duplicidades integrais:",
      bronze.groupBy(*df_raw.columns).count().filter("count > 1").count())
print("Países duplicados:",
      bronze.groupBy("Country").count().filter("count > 1").count())

Country,Region,Population (millions),HDI,GDP per Capita,Cropland Footprint,Grazing Footprint,Forest Footprint,Carbon Footprint,Fish Footprint,Total Ecological Footprint,Cropland,Grazing Land,Forest Land,Fishing Water,Urban Land,Total Biocapacity,Biocapacity Deficit or Reserve,Earths Required,Countries Required,Data Quality
0,0,0,16,15,15,15,15,15,15,0,15,15,15,15,15,0,0,0,0,0


Registros Bronze: 188
Duplicidades integrais: 0
Países duplicados: 0


## 4. Silver — limpeza e padronização

A Silver é construída com PySpark a partir da **Bronze persistida**. Ela mantém registros incompletos para rastreabilidade e adiciona indicadores de qualidade.

In [0]:
rename_map = {
    "Country":"country","Region":"region","Population (millions)":"population_millions",
    "HDI":"hdi","GDP per Capita":"gdp_per_capita_raw",
    "Cropland Footprint":"cropland_footprint","Grazing Footprint":"grazing_footprint",
    "Forest Footprint":"forest_footprint","Carbon Footprint":"carbon_footprint",
    "Fish Footprint":"fish_footprint","Total Ecological Footprint":"total_ecological_footprint",
    "Cropland":"cropland","Grazing Land":"grazing_land","Forest Land":"forest_land",
    "Fishing Water":"fishing_water","Urban Land":"urban_land",
    "Total Biocapacity":"total_biocapacity",
    "Biocapacity Deficit or Reserve":"biocapacity_deficit_or_reserve",
    "Earths Required":"earths_required","Countries Required":"countries_required",
    "Data Quality":"data_quality"
}

silver = spark.table(BRONZE_TABLE)
for old,new in rename_map.items():
    silver = silver.withColumnRenamed(old,new)

silver = silver.withColumn(
    "gdp_per_capita",
    F.regexp_replace(F.col("gdp_per_capita_raw").cast("string"), r"[$,]", "").cast("double")
)

numeric_cols = [
    "population_millions","hdi","cropland_footprint","grazing_footprint",
    "forest_footprint","carbon_footprint","fish_footprint",
    "total_ecological_footprint","cropland","grazing_land","forest_land",
    "fishing_water","urban_land","total_biocapacity",
    "biocapacity_deficit_or_reserve","earths_required","countries_required"
]
for c in numeric_cols:
    silver = silver.withColumn(c, F.col(c).cast("double"))

analysis_cols = ["hdi","gdp_per_capita","carbon_footprint","fish_footprint",
                 "cropland_footprint","grazing_footprint"]
complete = F.lit(True)
for c in analysis_cols:
    complete = complete & F.col(c).isNotNull()

silver = (
    silver.dropDuplicates()
    .withColumn("is_complete_for_analysis", complete)
    .withColumn("quality_rule_hdi",
        F.when(F.col("hdi").isNull(),"MISSING")
         .when((F.col("hdi") < 0) | (F.col("hdi") > 1),"INVALID").otherwise("OK"))
    .withColumn("quality_rule_footprints",
        F.when((F.col("carbon_footprint") < 0) | (F.col("fish_footprint") < 0) |
               (F.col("cropland_footprint") < 0) | (F.col("grazing_footprint") < 0),
               "INVALID").otherwise("OK"))
    .withColumn("silver_processed_timestamp", F.current_timestamp())
)

(silver.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true").saveAsTable(SILVER_TABLE))

print("Silver:", SILVER_TABLE, "| registros:", spark.table(SILVER_TABLE).count())

Silver: workspace.mvp_ecological_footprint.countries_silver | registros: 188


## 5. Qualidade — SQL sobre a Silver

Estas consultas geram evidências diretamente da tabela armazenada.

In [0]:
display(spark.sql(f'''
SELECT
  COUNT(*) AS total_registros,
  SUM(CASE WHEN hdi IS NULL THEN 1 ELSE 0 END) AS hdi_nulos,
  SUM(CASE WHEN gdp_per_capita IS NULL THEN 1 ELSE 0 END) AS gdp_nulos,
  SUM(CASE WHEN carbon_footprint IS NULL THEN 1 ELSE 0 END) AS carbon_nulos,
  SUM(CASE WHEN fish_footprint IS NULL THEN 1 ELSE 0 END) AS fish_nulos,
  SUM(CASE WHEN cropland_footprint IS NULL THEN 1 ELSE 0 END) AS cropland_nulos,
  SUM(CASE WHEN grazing_footprint IS NULL THEN 1 ELSE 0 END) AS grazing_nulos,
  SUM(CASE WHEN is_complete_for_analysis THEN 1 ELSE 0 END) AS completos_analise
FROM {SILVER_TABLE}
'''))

display(spark.sql(f'''
SELECT quality_rule_hdi, quality_rule_footprints, COUNT(*) AS registros
FROM {SILVER_TABLE}
GROUP BY quality_rule_hdi, quality_rule_footprints
ORDER BY registros DESC
'''))

total_registros,hdi_nulos,gdp_nulos,carbon_nulos,fish_nulos,cropland_nulos,grazing_nulos,completos_analise
188,16,15,15,15,15,15,162


quality_rule_hdi,quality_rule_footprints,registros
OK,OK,172
MISSING,OK,16


## 6. Gold detalhada

A Gold principal contém apenas registros adequados às perguntas analíticas. Ela é a **fonte oficial das análises** posteriores.

In [0]:
gold = (
    spark.table(SILVER_TABLE)
    .filter("is_complete_for_analysis = true")
    .filter("quality_rule_hdi = 'OK'")
    .filter("quality_rule_footprints = 'OK'")
    .select("country","region","population_millions","hdi","gdp_per_capita",
            "carbon_footprint","fish_footprint","cropland_footprint",
            "grazing_footprint","data_quality")
    .withColumn("hdi_band",
        F.when(F.col("hdi") < .550,"Baixo")
         .when(F.col("hdi") < .700,"Médio")
         .when(F.col("hdi") < .800,"Alto").otherwise("Muito alto"))
    .withColumn("gdp_per_capita_band",
        F.when(F.col("gdp_per_capita") < 2000,"< 2 mil")
         .when(F.col("gdp_per_capita") < 10000,"2–10 mil")
         .when(F.col("gdp_per_capita") < 30000,"10–30 mil").otherwise(">= 30 mil"))
    .withColumn("gold_processed_timestamp", F.current_timestamp())
)

(gold.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true").saveAsTable(GOLD_TABLE))

print("Gold principal:", GOLD_TABLE, "| registros:", spark.table(GOLD_TABLE).count())
display(spark.table(GOLD_TABLE).limit(10))

Gold principal: workspace.mvp_ecological_footprint.countries_environmental_analysis | registros: 162


country,region,population_millions,hdi,gdp_per_capita,carbon_footprint,fish_footprint,cropland_footprint,grazing_footprint,data_quality,hdi_band,gdp_per_capita_band,gold_processed_timestamp
Bahamas,Latin America,0.37,0.78,22647.3,4.46,0.14,0.97,1.05,3L,Alto,10–30 mil,2026-09-23T13:20:52.885Z
Ecuador,Latin America,15.49,0.73,5192.88,1.08,0.18,0.3,0.3,5,Alto,2–10 mil,2026-09-23T13:20:52.885Z
Luxembourg,European Union,0.52,0.89,114665.0,12.65,0.13,1.1,0.76,5,Muito alto,>= 30 mil,2026-09-23T13:20:52.885Z
Nigeria,Africa,168.83,0.5,2612.12,0.21,0.08,0.53,0.1,5,Baixo,2–10 mil,2026-09-23T13:20:52.885Z
Qatar,Middle East/Central Asia,2.05,0.85,99431.5,9.57,0.19,0.57,0.27,3L,Muito alto,>= 30 mil,2026-09-23T13:20:52.885Z
Slovakia,European Union,5.45,0.84,18103.1,2.82,0.03,0.31,0.08,5,Muito alto,10–30 mil,2026-09-23T13:20:52.885Z
Thailand,Asia-Pacific,66.78,0.72,5479.29,1.54,0.13,0.67,0.02,6,Alto,2–10 mil,2026-09-23T13:20:52.885Z
Austria,European Union,8.46,0.88,51274.1,4.14,0.06,0.82,0.27,5,Muito alto,>= 30 mil,2026-09-23T13:20:52.885Z
Bahrain,Middle East/Central Asia,1.32,0.82,24299.0,6.19,0.07,0.52,0.45,3L,Muito alto,10–30 mil,2026-09-23T13:20:52.885Z
Denmark,European Union,5.6,0.92,61413.6,2.6,0.24,1.18,0.47,5,Muito alto,>= 30 mil,2026-09-23T13:20:52.885Z


## 7. Tabelas auxiliares Gold para análise

Para que a maior parte da análise ocorra em dados já armazenados no Databricks, o pipeline cria tabelas Gold auxiliares:

- `analysis_by_region`
- `analysis_by_hdi_band`
- `analysis_by_gdp_band`
- `analysis_correlations`
- `analysis_outliers`

Essas tabelas são produtos analíticos persistidos, não DataFrames temporários.

In [0]:
spark.sql(f'''
CREATE OR REPLACE TABLE {GOLD_REGION_TABLE}
USING DELTA AS
SELECT
  region,
  COUNT(*) AS countries,
  ROUND(AVG(hdi),3) AS avg_hdi,
  ROUND(AVG(gdp_per_capita),2) AS avg_gdp_per_capita,
  ROUND(AVG(carbon_footprint),3) AS avg_carbon_footprint,
  ROUND(AVG(fish_footprint),3) AS avg_fish_footprint,
  ROUND(AVG(cropland_footprint),3) AS avg_cropland_footprint,
  ROUND(AVG(grazing_footprint),3) AS avg_grazing_footprint
FROM {GOLD_TABLE}
GROUP BY region
''')

spark.sql(f'''
CREATE OR REPLACE TABLE {GOLD_HDI_TABLE}
USING DELTA AS
SELECT
  hdi_band,
  COUNT(*) AS countries,
  ROUND(AVG(hdi),3) AS avg_hdi,
  ROUND(AVG(gdp_per_capita),2) AS avg_gdp_per_capita,
  ROUND(AVG(carbon_footprint),3) AS avg_carbon_footprint,
  ROUND(AVG(fish_footprint),3) AS avg_fish_footprint,
  ROUND(AVG(cropland_footprint),3) AS avg_cropland_footprint,
  ROUND(AVG(grazing_footprint),3) AS avg_grazing_footprint
FROM {GOLD_TABLE}
GROUP BY hdi_band
''')

spark.sql(f'''
CREATE OR REPLACE TABLE {GOLD_GDP_TABLE}
USING DELTA AS
SELECT
  gdp_per_capita_band,
  COUNT(*) AS countries,
  ROUND(AVG(hdi),3) AS avg_hdi,
  ROUND(AVG(gdp_per_capita),2) AS avg_gdp_per_capita,
  ROUND(AVG(carbon_footprint),3) AS avg_carbon_footprint,
  ROUND(AVG(fish_footprint),3) AS avg_fish_footprint,
  ROUND(AVG(cropland_footprint),3) AS avg_cropland_footprint,
  ROUND(AVG(grazing_footprint),3) AS avg_grazing_footprint
FROM {GOLD_TABLE}
GROUP BY gdp_per_capita_band
''')

print("Tabelas Gold agregadas criadas.")

Tabelas Gold agregadas criadas.


In [0]:
# Correlações calculadas pelo Spark sobre a Gold persistida e persistidas em outra Gold.
pairs = [
    ("hdi","carbon_footprint"), ("gdp_per_capita","carbon_footprint"),
    ("hdi","fish_footprint"), ("gdp_per_capita","fish_footprint"),
    ("hdi","cropland_footprint"), ("gdp_per_capita","cropland_footprint"),
    ("hdi","grazing_footprint"), ("gdp_per_capita","grazing_footprint")
]

gold_df = spark.table(GOLD_TABLE)
corr_rows = []
for x,y in pairs:
    value = gold_df.stat.corr(x,y,method="pearson")
    corr_rows.append((x,y,float(value),"pearson"))

corr_df = spark.createDataFrame(corr_rows, ["variable_x","variable_y","correlation","method"])
(corr_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true").saveAsTable(GOLD_CORR_TABLE))

display(spark.table(GOLD_CORR_TABLE).orderBy(F.desc(F.abs("correlation"))))

variable_x,variable_y,correlation,method
gdp_per_capita,carbon_footprint,0.823759745985214,pearson
hdi,carbon_footprint,0.6990762160228162,pearson
hdi,cropland_footprint,0.5672268246053908,pearson
gdp_per_capita,cropland_footprint,0.5061333237596535,pearson
hdi,fish_footprint,0.2084325258213821,pearson
gdp_per_capita,fish_footprint,0.15296246841501138,pearson
gdp_per_capita,grazing_footprint,0.10331762264497407,pearson
hdi,grazing_footprint,0.09111492557678914,pearson


In [0]:
# Outliers: limites IQR calculados pelo Spark e resultados persistidos.
targets = ["gdp_per_capita","carbon_footprint","fish_footprint",
           "cropland_footprint","grazing_footprint"]

outlier_parts = []
g = spark.table(GOLD_TABLE)

for col in targets:
    q1, q3 = g.approxQuantile(col, [0.25,0.75], 0.0)
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    part = (
        g.filter((F.col(col) < low) | (F.col(col) > high))
         .select("country","region",
                 F.lit(col).alias("metric"),
                 F.col(col).alias("metric_value"),
                 F.lit(float(low)).alias("lower_limit"),
                 F.lit(float(high)).alias("upper_limit"))
    )
    outlier_parts.append(part)

outliers = outlier_parts[0]
for p in outlier_parts[1:]:
    outliers = outliers.unionByName(p)

(outliers.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true").saveAsTable(GOLD_OUTLIER_TABLE))

display(spark.table(GOLD_OUTLIER_TABLE).orderBy("metric", F.desc("metric_value")))

country,region,metric,metric_value,lower_limit,upper_limit
Luxembourg,European Union,carbon_footprint,12.65,-2.95,5.93
Qatar,Middle East/Central Asia,carbon_footprint,9.57,-2.95,5.93
Trinidad and Tobago,Latin America,carbon_footprint,6.89,-2.95,5.93
Kuwait,Middle East/Central Asia,carbon_footprint,6.89,-2.95,5.93
United Arab Emirates,Middle East/Central Asia,carbon_footprint,6.37,-2.95,5.93
Bahrain,Middle East/Central Asia,carbon_footprint,6.19,-2.95,5.93
Australia,Asia-Pacific,cropland_footprint,2.68,-0.20499999999999996,1.275
Latvia,European Union,cropland_footprint,2.28,-0.20499999999999996,1.275
Lithuania,European Union,cropland_footprint,1.89,-0.20499999999999996,1.275
Sweden,European Union,cropland_footprint,1.47,-0.20499999999999996,1.275


## 8. Análise final — SQL sobre tabelas armazenadas

As próximas células **não analisam DataFrames locais**. Elas consultam as tabelas Gold persistidas no Databricks.

In [0]:
# 8.1 Relações entre desenvolvimento e footprints
display(spark.sql(f'''
SELECT
  variable_x,
  variable_y,
  ROUND(correlation, 3) AS pearson_correlation
FROM {GOLD_CORR_TABLE}
ORDER BY ABS(correlation) DESC
'''))

variable_x,variable_y,pearson_correlation
gdp_per_capita,carbon_footprint,0.824
hdi,carbon_footprint,0.699
hdi,cropland_footprint,0.567
gdp_per_capita,cropland_footprint,0.506
hdi,fish_footprint,0.208
gdp_per_capita,fish_footprint,0.153
gdp_per_capita,grazing_footprint,0.103
hdi,grazing_footprint,0.091


In [0]:
# 8.2 Perfil por região
display(spark.sql(f'''
SELECT *
FROM {GOLD_REGION_TABLE}
ORDER BY avg_hdi DESC
'''))

region,countries,avg_hdi,avg_gdp_per_capita,avg_carbon_footprint,avg_fish_footprint,avg_cropland_footprint,avg_grazing_footprint
North America,2,0.91,50935.2,5.45,0.12,1.295,0.315
European Union,24,0.865,35959.72,3.264,0.116,0.983,0.26
Northern/Eastern Europe,11,0.775,14209.65,2.211,0.06,0.708,0.16
Middle East/Central Asia,22,0.732,16368.72,3.046,0.067,0.602,0.217
Latin America,29,0.718,8319.64,1.618,0.131,0.489,0.414
Asia-Pacific,27,0.693,12611.26,1.503,0.215,0.602,0.248
Africa,47,0.514,2705.44,0.541,0.095,0.397,0.234


In [0]:
# 8.3 Perfil por faixa de HDI
display(spark.sql(f'''
SELECT *
FROM {GOLD_HDI_TABLE}
ORDER BY
  CASE hdi_band
    WHEN 'Baixo' THEN 1 WHEN 'Médio' THEN 2
    WHEN 'Alto' THEN 3 WHEN 'Muito alto' THEN 4
  END
'''))

hdi_band,countries,avg_hdi,avg_gdp_per_capita,avg_carbon_footprint,avg_fish_footprint,avg_cropland_footprint,avg_grazing_footprint
Baixo,40,0.461,1035.68,0.202,0.077,0.368,0.219
Médio,32,0.629,3932.62,0.951,0.092,0.456,0.231
Alto,48,0.747,8164.54,1.927,0.147,0.577,0.318
Muito alto,42,0.867,38835.29,3.869,0.146,0.936,0.279


In [0]:
# 8.4 Perfil por faixa de PIB per capita
display(spark.sql(f'''
SELECT *
FROM {GOLD_GDP_TABLE}
ORDER BY avg_gdp_per_capita
'''))

gdp_per_capita_band,countries,avg_hdi,avg_gdp_per_capita,avg_carbon_footprint,avg_fish_footprint,avg_cropland_footprint,avg_grazing_footprint
< 2 mil,50,0.496,968.79,0.285,0.077,0.379,0.184
2–10 mil,57,0.701,5313.1,1.331,0.114,0.552,0.299
10–30 mil,30,0.796,16864.54,2.97,0.156,0.726,0.307
>= 30 mil,25,0.889,53321.15,4.573,0.167,0.966,0.306


In [0]:
# 8.5 Países com maior Carbon Footprint
display(spark.sql(f'''
SELECT country, region, hdi, gdp_per_capita, carbon_footprint
FROM {GOLD_TABLE}
ORDER BY carbon_footprint DESC
LIMIT 20
'''))

country,region,hdi,gdp_per_capita,carbon_footprint
Luxembourg,European Union,0.89,114665.0,12.65
Qatar,Middle East/Central Asia,0.85,99431.5,9.57
Trinidad and Tobago,Latin America,0.77,18310.8,6.89
Kuwait,Middle East/Central Asia,0.82,41830.5,6.89
United Arab Emirates,Middle East/Central Asia,0.83,40817.4,6.37
Bahrain,Middle East/Central Asia,0.82,24299.0,6.19
Singapore,Asia-Pacific,0.91,53122.4,5.91
United States of America,North America,0.91,49725.0,5.9
Oman,Middle East/Central Asia,0.79,22622.8,5.8
Canada,North America,0.91,52145.4,5.0


In [0]:
# 8.6 Países com maior Fish, Cropland e Grazing Footprint
display(spark.sql(f'''
SELECT country, region, fish_footprint
FROM {GOLD_TABLE}
ORDER BY fish_footprint DESC
LIMIT 15
'''))

display(spark.sql(f'''
SELECT country, region, cropland_footprint
FROM {GOLD_TABLE}
ORDER BY cropland_footprint DESC
LIMIT 15
'''))

display(spark.sql(f'''
SELECT country, region, grazing_footprint
FROM {GOLD_TABLE}
ORDER BY grazing_footprint DESC
LIMIT 15
'''))

country,region,fish_footprint
Saint Kitts and Nevis,Latin America,0.81
Papua New Guinea,Asia-Pacific,0.73
Namibia,Africa,0.72
New Zealand,Asia-Pacific,0.7
Fiji,Asia-Pacific,0.62
Mauritius,Africa,0.55
Sao Tome and Principe,Africa,0.47
Solomon Islands,Asia-Pacific,0.46
Samoa,Asia-Pacific,0.42
"Korea, Republic of",Asia-Pacific,0.41


country,region,cropland_footprint
Australia,Asia-Pacific,2.68
Latvia,European Union,2.28
Lithuania,European Union,1.89
Sweden,European Union,1.47
Canada,North America,1.46
Belarus,Northern/Eastern Europe,1.32
France,European Union,1.23
Tonga,Asia-Pacific,1.19
Denmark,European Union,1.18
Belgium,European Union,1.15


country,region,grazing_footprint
Mongolia,Asia-Pacific,3.47
Bolivia,Latin America,1.69
Mauritania,Africa,1.2
Paraguay,Latin America,1.1
Bahamas,Latin America,1.05
Uruguay,Latin America,0.98
Botswana,Africa,0.89
Brazil,Latin America,0.85
Argentina,Latin America,0.79
Luxembourg,European Union,0.76


In [0]:
# 8.7 Outliers persistidos
display(spark.sql(f'''
SELECT metric, COUNT(*) AS outlier_count
FROM {GOLD_OUTLIER_TABLE}
GROUP BY metric
ORDER BY outlier_count DESC
'''))

display(spark.sql(f'''
SELECT *
FROM {GOLD_OUTLIER_TABLE}
ORDER BY metric, metric_value DESC
'''))

metric,outlier_count
gdp_per_capita,23
fish_footprint,15
grazing_footprint,12
carbon_footprint,6
cropland_footprint,6


country,region,metric,metric_value,lower_limit,upper_limit
Luxembourg,European Union,carbon_footprint,12.65,-2.95,5.93
Qatar,Middle East/Central Asia,carbon_footprint,9.57,-2.95,5.93
Trinidad and Tobago,Latin America,carbon_footprint,6.89,-2.95,5.93
Kuwait,Middle East/Central Asia,carbon_footprint,6.89,-2.95,5.93
United Arab Emirates,Middle East/Central Asia,carbon_footprint,6.37,-2.95,5.93
Bahrain,Middle East/Central Asia,carbon_footprint,6.19,-2.95,5.93
Australia,Asia-Pacific,cropland_footprint,2.68,-0.20499999999999996,1.275
Latvia,European Union,cropland_footprint,2.28,-0.20499999999999996,1.275
Lithuania,European Union,cropland_footprint,1.89,-0.20499999999999996,1.275
Sweden,European Union,cropland_footprint,1.47,-0.20499999999999996,1.275


## 9. Visualizações no Databricks

Visualizações **a partir das saídas SQL** das células anteriores usando o recurso de visualização do próprio Databricks.

Gráficos:
1. dispersão: `hdi` × `carbon_footprint` consultando a Gold principal;
2. dispersão: `gdp_per_capita` × `carbon_footprint` consultando a Gold principal.

In [0]:
# Query-base para gráfico de dispersão no Databricks
display(spark.sql(f'''
SELECT hdi, carbon_footprint, country, region
FROM {GOLD_TABLE}
ORDER BY hdi
'''))

display(spark.sql(f'''
SELECT gdp_per_capita, carbon_footprint, country, region
FROM {GOLD_TABLE}
ORDER BY gdp_per_capita
'''))

hdi,carbon_footprint,country,region
0.34,0.1,Niger,Africa
0.37,0.08,Central African Republic,Africa
0.39,0.03,Eritrea,Africa
0.39,0.12,Burkina Faso,Africa
0.39,0.01,Chad,Africa
0.39,0.04,Burundi,Africa
0.4,0.07,Sierra Leone,Africa
0.41,0.16,Guinea,Africa
0.41,0.1,Mali,Africa
0.41,0.17,Mozambique,Africa


Databricks visualization. Run in Databricks to view.

gdp_per_capita,carbon_footprint,country,region
276.69,0.04,Burundi,Africa
338.63,0.07,"Congo, Democratic Republic of",Africa
379.38,0.07,Ethiopia,Africa
397.38,0.14,Liberia,Africa
410.91,0.1,Niger,Africa
439.73,0.03,Eritrea,Africa
456.33,0.07,Madagascar,Africa
459.09,0.16,Guinea,Africa
493.84,0.07,Malawi,Africa
495.04,0.08,Central African Republic,Africa


Databricks visualization. Run in Databricks to view.

## 10. Catálogo físico do projeto

**Bronze**
- `countries_bronze`

**Silver**
- `countries_silver`

**Gold**
- `countries_environmental_analysis`
- `analysis_by_region`
- `analysis_by_hdi_band`
- `analysis_by_gdp_band`
- `analysis_correlations`
- `analysis_outliers`

A existência dessas tabelas demonstra que o resultado analítico foi materializado no Lakehouse e pode ser consumido por SQL, dashboards ou outros processos sem repetir toda a preparação.

In [0]:
# Inventário das tabelas do MVP
display(spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}"))

database,tableName,isTemporary
mvp_ecological_footprint,analysis_by_gdp_band,false
mvp_ecological_footprint,analysis_by_hdi_band,false
mvp_ecological_footprint,analysis_by_region,false
mvp_ecological_footprint,analysis_correlations,false
mvp_ecological_footprint,analysis_outliers,false
mvp_ecological_footprint,countries_bronze,false
mvp_ecological_footprint,countries_environmental_analysis,false
mvp_ecological_footprint,countries_silver,false


## 11. Autoavaliação

O pipeline utiliza PySpark como ferramenta essencial de engenharia para ingestão, limpeza, regras de qualidade e materialização das camadas. Entretanto, a análise final foi deliberadamente organizada sobre **tabelas Delta persistidas no Databricks**, aproximando o MVP de um fluxo real de Engenharia de Dados.

A Gold principal funciona como base analítica detalhada, enquanto tabelas Gold auxiliares materializam agregações, correlações e outliers. Isso melhora reuso, rastreabilidade e clareza entre transformação e consumo.

A principal limitação permanece sendo o caráter transversal da base: as relações identificadas são associações entre países e não evidências de causalidade.